# Chapter 7: Tool Manipulation and Orchestration Agents

**Book:** *30 Agents Every AI Engineer Must Build*  
**Author:** Imran Ahmad  
**Publisher:** Packt Publishing, 2026  
**Chapter Pages:** pp. 173–201

---

> *"The best way to predict the future is to invent it."*
> — Alan Kay, computer scientist

## Introduction

While language models provide the reasoning substrate for understanding user goals, it is the **use of tools** that enables agents to execute tasks and affect the real world. This chapter establishes the architectural foundation for understanding how agents translate reasoning into action through three progressive patterns:

1. **Tool-Using Agents** (§7.1–7.3, pp. 174–186) — A single agent extends its capabilities by invoking external functions through a Think/Plan/Act cycle. Covers tool registries with schema contracts, the three-stage selection funnel (intent classification → semantic search → constraint filtering), and layered error handling with safe wrappers and fallback chains.

2. **Chain-of-Agents Orchestrators** (§7.4–7.6, pp. 186–194) — Multiple specialized agents collaborate under a cooperation protocol with four architectural pillars: defined roles, common communication, shared memory (episodic + semantic), and execution orchestration. Includes conflict resolution via automated arbitration with confidence-based consensus.

3. **Agentic Workflow Systems** (§7.7, pp. 195–201) — Stateful business processes modeled as state machines with human-in-the-loop checkpoints, guard conditions, and audit trails. Two case studies: e-commerce order processing and multi-agent insurance claims.

### Key Architectural Insight

Tool invocation represents the fundamental mechanism that bridges cognitive operations and concrete actions, transforming an agent from a simple conversationalist into a functional system component. The architectural challenge lies in creating systems in which tool invocation becomes a **seamless extension of agent cognition** rather than a brittle integration point.

A robust Tool-Using agent architecture separates four concerns: a **reasoning core** (Think/Plan), a **tool registry** with explicit schema contracts, an **execution engine** managing state and retries, and a **guarded tool chest** of modular functions with single responsibilities.

### Fail-Gracefully Architecture

This notebook is built around a Fail-Gracefully architecture: if no API key is present, the system automatically enters **Simulation Mode**, returning chapter-derived mock data through a context-aware `MockLLM`. Every agent action produces color-coded log output. Every tool invocation is wrapped in defensive logic. The notebook always runs to completion.

---

---
## Section 0: Setup and Configuration

This section loads dependencies, detects the operating mode (Simulation vs. Live),
and initializes the shared infrastructure used by all subsequent sections.

**Key components initialized here:**
- `helpers` package (color logger, resilience decorators, MockLLM)
- `LIVE_MODE` and `INTERACTIVE_MODE` flags
- `AgentError` exception class used by workflow sections

In [ ]:
# =============================================================================
# Section 0: Setup and Configuration
# Chapter 7: Tool Manipulation and Orchestration Agents
# Book: "Agents" by Imran Ahmad (Packt, 2026 — B34135)
# =============================================================================

import os
import sys
import json
import time
import getpass
from typing import Dict, Any, List, Tuple, Optional

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Ensure helpers/ is importable from the notebook's working directory
sys.path.insert(0, os.path.abspath("."))

from helpers import (
    log_info, log_success, log_error, log_warning, log_mock, log_step,
    graceful_fallback, safe_invoke,
    MockLLM,
)

# Matplotlib backend for inline rendering
%matplotlib inline

# ---------------------------------------------------------------------------
# Custom Exception — used by workflow sections (Sections 7.7, 7.7b)
# ---------------------------------------------------------------------------
class AgentError(Exception):
    """Raised when a workflow step fails critically and the pipeline must stop."""
    pass

log_info("Core imports complete. helpers package loaded.")

In [ ]:
# =============================================================================
# Section 0: API Key Detection and Mode Selection
# Ref: Strategy §3.4 — Simulation Mode Activation Flow
# =============================================================================

# --- Step 1: Try .env file ---
try:
    from dotenv import load_dotenv
    load_dotenv()
    log_info("python-dotenv loaded. Checking .env for OPENAI_API_KEY...")
except ImportError:
    log_warning("python-dotenv not installed. Skipping .env loading.")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()

# --- Step 2: Interactive fallback via getpass ---
# Set False for Colab, CI/CD, nbconvert --execute, or non-interactive envs
INTERACTIVE_MODE = True

if not OPENAI_API_KEY and INTERACTIVE_MODE:
    try:
        OPENAI_API_KEY = getpass.getpass(
            "Enter OpenAI API key (or press Enter to skip): "
        ).strip()
    except (EOFError, OSError, Exception):
        # Catches StdinNotImplementedError, EOFError, and any other input failure
        OPENAI_API_KEY = ""
        INTERACTIVE_MODE = False
        log_warning("Non-interactive environment detected. INTERACTIVE_MODE set to False.")

# --- Step 3: Determine operating mode ---
LIVE_MODE = bool(OPENAI_API_KEY) and OPENAI_API_KEY != "your-key-here"

if LIVE_MODE:
    log_success("LIVE MODE active. LLM calls will use the OpenAI API.")
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    llm = None  # Will be initialized with OpenAI client when needed
else:
    llm = MockLLM(verbose=True)
    log_mock("╔══════════════════════════════════════════════════════════╗")
    log_mock("║  SIMULATION MODE ACTIVE                                 ║")
    log_mock("║  All LLM calls return chapter-derived mock responses.   ║")
    log_mock("║  To use a live API, create .env from .env.template      ║")
    log_mock("╚══════════════════════════════════════════════════════════╝")

# --- Summary ---
log_info(f"Operating Mode: {'LIVE' if LIVE_MODE else 'SIMULATION'}")
log_info(f"Interactive Mode: {INTERACTIVE_MODE}")
log_info(f"Python: {sys.version.split()[0]}")
log_info(f"pandas: {pd.__version__}, matplotlib: {matplotlib.__version__}")

---
## Section 1: The Tool-Using Agent

**Chapter Reference:** Section 7.1 — Agents in Action: Tool Invocation Patterns

A Tool-Using agent extends its core reasoning by invoking a predefined set of external
functions. The architecture consists of four primary components (Figure 7.1):

1. **Reasoning Core** — Parses intent and formulates a plan (Think + Plan)
2. **Tool Registry** — Catalog of available tools with metadata and schemas
3. **Execution Engine** — Manages state, retries, and error handling (Act)
4. **Tool Chest** — The collection of guarded functions the agent can execute

This section implements the data visualization assistant's tool chest: four composable
functions that divide the visualization pipeline into clear, testable steps.

#### Figure 7.1 — Internal Architecture of Tool-Using Agents *(Book p. 176)*

```
                                    ┌───────────────────────┐
  User Input ──────────────────────▶│    Reasoning Core     │
  "Show chart"                      │  ┌─────────────────┐  │    3. Pass Plan
                                    │  │ Intent Parsing   │  │───────────────┐
               1. Discover          │  │ Planning         │  │               │
               & Select Tools       │  └─────────────────┘  │               ▼
                    │               └───────────▲───────────┘    ┌─────────────────┐
                    ▼                           │                │ Execution Engine │
           ┌─────────────────┐     2. Return    │                │ State Management │
           │  Tool Registry  │──── Candidates ──┘                │ Error Handling   │
           │  Metadata       │                                   └────────┬────────┘
           │  Schemas        │                                   4. Invoke│Tool
           │  Status         │                                            ▼
           └─────────────────┘                        ┌──────────────────────────────┐
                                                      │        Tool Chest             │
                                                      │  ┌──────────┐ ┌──────────┐   │
                                                      │  │ load_csv │→│plot_chart │   │
                                                      │  └──────────┘ └──────────┘   │
                                                      │  ┌──────────┐ ┌──────────┐   │
                                                      │  │aggregate │ │ validate │   │
                                                      │  └──────────┘ └──────────┘   │
                                                      └──────────────┬───────────────┘
                                                                     ▼
                                                            Generated Chart
```

This architecture cleanly separates decision logic from execution. Explicit schemas make interfaces predictable, while safety wrappers provide resilience. The same structure supports both reactive queries (single tool) and deliberative plans (chained tools).


In [ ]:
# =============================================================================
# Section 1: Tool Chest — Four Composable Visualization Functions
# Ref: Section 7.1, pp. 178–179 — Implementation Example
#
# Each function has a single responsibility (single responsibility principle):
#   load_csv         — data ingestion
#   group_by_and_aggregate — data summarization
#   plot_bar_chart   — bar chart rendering
#   plot_line_chart  — line chart rendering
#
# All functions are wrapped with @graceful_fallback for resilience (Section 7.3).
# =============================================================================

# --- Tool 1: Data Ingestion ---
# Ref: Section 7.1, p. 6 — "Handles data ingestion"

@graceful_fallback(fallback_return=pd.DataFrame(), section="7.1")
def load_csv(file_path: str) -> pd.DataFrame:
    """Loads data from a specified CSV file into a pandas DataFrame."""
    df = pd.read_csv(file_path)
    log_step("7.1", 1, f"Loaded {len(df)} rows from '{file_path}'.")
    return df


# --- Tool 2: Data Transformation and Summarization ---
# Ref: Section 7.1, p. 7 — "Performs data transformation and summarization"

@graceful_fallback(fallback_return=None, section="7.1")
def group_by_and_aggregate(
    df: pd.DataFrame,
    group_by_col: str,
    agg_col: str,
    agg_func: str = "sum",
) -> pd.DataFrame:
    """Groups the DataFrame and applies an aggregation function."""
    if group_by_col not in df.columns or agg_col not in df.columns:
        raise ValueError(
            f"Column not found in DataFrame. "
            f"Available: {list(df.columns)}"
        )
    func = {"sum": "sum", "mean": "mean", "count": "count"}.get(agg_func, "sum")
    result = df.groupby(group_by_col)[agg_col].agg(func).reset_index()
    log_step("7.1", 2, f"Grouped '{agg_col}' by '{group_by_col}' using '{func}'.")
    return result


# --- Tool 3: Bar Chart Rendering ---
# Ref: Section 7.1, p. 7 — "Handles final rendering for a bar chart"

@graceful_fallback(fallback_return=None, section="7.1")
def plot_bar_chart(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    title: str,
    output_path: str,
) -> str:
    """Generates and saves a bar chart from the data."""
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(df[x_col].astype(str), df[y_col], color="#4A90D9")
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=100)
    plt.show()
    plt.close(fig)
    log_step("7.1", 3, f"Saved bar chart to '{output_path}'.")
    return output_path


# --- Tool 4: Line Chart Rendering ---
# Ref: Section 7.1, p. 7-8 — "Handles final rendering for a line chart"

@graceful_fallback(fallback_return=None, section="7.1")
def plot_line_chart(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    title: str,
    output_path: str,
) -> str:
    """Generates and saves a line chart from the data."""
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(df[x_col].astype(str), df[y_col], marker="o", color="#4A90D9")
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=100)
    plt.show()
    plt.close(fig)
    log_step("7.1", 4, f"Saved line chart to '{output_path}'.")
    return output_path


log_success("Tool chest defined: load_csv, group_by_and_aggregate, plot_bar_chart, plot_line_chart.")

In [ ]:
# =============================================================================
# Section 1: Tool Registry
# Ref: Section 7.1, pp. 175–177 — "The Tool Registry"
#
# The registry maintains critical metadata for each tool: name, description,
# input/output schemas, and operational status. These schemas serve as explicit,
# reliable contracts between the reasoning core and the execution layer.
# =============================================================================

TOOL_REGISTRY = {
    "load_csv": {
        "function": load_csv,
        "description": "Loads data from a specified CSV file into a pandas DataFrame.",
        "input_schema": {"file_path": "str"},
        "output_schema": "pd.DataFrame",
        "status": "active",
    },
    "group_by_and_aggregate": {
        "function": group_by_and_aggregate,
        "description": "Groups a DataFrame by a column and applies an aggregation function.",
        "input_schema": {
            "df": "pd.DataFrame",
            "group_by_col": "str",
            "agg_col": "str",
            "agg_func": "str (sum|mean|count)",
        },
        "output_schema": "pd.DataFrame",
        "status": "active",
    },
    "plot_bar_chart": {
        "function": plot_bar_chart,
        "description": "Generates and saves a bar chart from aggregated data.",
        "input_schema": {
            "df": "pd.DataFrame",
            "x_col": "str",
            "y_col": "str",
            "title": "str",
            "output_path": "str",
        },
        "output_schema": "str (file path)",
        "status": "active",
    },
    "plot_line_chart": {
        "function": plot_line_chart,
        "description": "Generates and saves a line chart with markers from aggregated data.",
        "input_schema": {
            "df": "pd.DataFrame",
            "x_col": "str",
            "y_col": "str",
            "title": "str",
            "output_path": "str",
        },
        "output_schema": "str (file path)",
        "status": "active",
    },
}

# Display the registry for inspection
log_info("Tool Registry initialized:")
for name, meta in TOOL_REGISTRY.items():
    log_info(f"  {name}: {meta['description']} [status={meta['status']}]")

In [ ]:
# =============================================================================
# Section 1: Demo Pipeline — End-to-End Data Visualization
# Ref: Section 7.1, pp. 177–179 — Tool Chest composability
#
# Demonstrates the four tools working together in sequence:
#   1. load_csv → ingest the ad campaign dataset
#   2. group_by_and_aggregate → summarize spend by campaign
#   3. plot_bar_chart → render the bar chart
#   4. group_by_and_aggregate → summarize clicks over time
#   5. plot_line_chart → render the line chart
# =============================================================================

log_info("=" * 60)
log_info("Section 1 Demo: Data Visualization Pipeline")
log_info("=" * 60)

# Ensure outputs directory exists
os.makedirs("outputs", exist_ok=True)

# Step 1: Load the dataset
df = load_csv("data/sample_ad_campaign.csv")
if not df.empty:
    log_info(f"Dataset preview ({len(df)} rows):")
    print(df.head())
    print()

    # Step 2a: Aggregate spend by campaign (for bar chart)
    spend_by_campaign = group_by_and_aggregate(df, "campaign_name", "spend", "sum")
    if spend_by_campaign is not None:
        print(spend_by_campaign)
        print()
        # Step 3: Render bar chart
        plot_bar_chart(
            spend_by_campaign,
            "campaign_name", "spend",
            "Total Spend by Campaign",
            "outputs/spend_by_campaign.png",
        )

    # Step 2b: Aggregate clicks over time (for line chart)
    clicks_over_time = group_by_and_aggregate(df, "date", "clicks", "sum")
    if clicks_over_time is not None:
        print(clicks_over_time)
        print()
        # Step 4: Render line chart
        plot_line_chart(
            clicks_over_time,
            "date", "clicks",
            "Total Clicks Over Time",
            "outputs/clicks_over_time.png",
        )

    log_success("Section 1 Demo complete. Charts saved to outputs/.")
else:
    log_error("Dataset is empty. Cannot proceed with demo.")

---
## Section 2: Tool Discovery and Selection

**Chapter Reference:** Section 7.2 — Tool Discovery and Selection Algorithms

As an agent's tool chest grows, it must reliably select the correct tool for each task.
This section implements the first stage of the **Selection Funnel** (Figure 7.2):

1. **Intent Classification (broad filtering)** — Map the user's query to a relevant category
2. **Semantic Search (candidate ranking)** — Rank remaining tools by similarity (architectural)
3. **Constraint Filtering (final verification)** — Enforce input types, permissions, and status

The `parse_query` function below implements **template-based intent classification**,
the most direct approach where keywords map to structured parameters.

#### Figure 7.2 — Tool Selection Funnel *(Book p. 180)*

The multi-stage funnel transforms a large tool registry into a single verified, executable choice:

```
  All Available Tools (~100 tools)
         │
         ▼
  ┌──────────────────────────────┐
  │  1. Intent Classification    │ ──▶ ~20 tools
  │     Map query to categories  │
  └──────────────┬───────────────┘
                 ▼
  ┌──────────────────────────────┐
  │  2. Semantic Similarity      │ ──▶ ~5 tools
  │     Embedding-based ranking  │
  └──────────────┬───────────────┘
                 ▼
         Similarity > 0.7?
         Yes ▼      No → Discard
  ┌──────────────────────────────┐
  │  3. Constraint Filtering     │ ──▶ ~2-3 tools
  │     Input types, permissions │
  └──────────────┬───────────────┘
                 ▼
       ✓ Optimal Tool Selected
         (1 tool, ready to execute)
```


In [ ]:
# =============================================================================
# Section 2: Intent Classification through Template Matching
# Ref: Section 7.2, pp. 182–183 — "Implementation Example: Intent Classification"
#
# parse_query() acts as a lightweight intent classifier. It scans for specific
# keywords and patterns to extract: a metric, a dimension, and a chart type.
# This implements the Template Matching selection strategy.
# =============================================================================

@graceful_fallback(fallback_return=None, section="7.2")
def parse_query(query: str) -> dict:
    """A simple natural language parser to extract intent from the user's query.
    Uses keyword matching to identify the metric, dimension, and chart type."""
    query = query.lower()  # Normalize for case-insensitive matching

    # --- Step 1: Identify the metric ---
    # A simple lookup to map known metric keywords to their column names
    metric_map = {"spend": "spend", "conversions": "conversions", "clicks": "clicks"}
    metric = next((metric_map[key] for key in metric_map if key in query), None)

    # --- Step 2: Identify the dimension and chart type ---
    # Use keywords to infer how the user wants to group and visualize data
    dimension, chart_type = None, None
    if "by campaign" in query or "which campaign" in query:
        dimension, chart_type = "campaign_name", "bar"
    elif "over time" in query or "trend" in query:
        dimension, chart_type = "date", "line"

    # --- Step 3: Validate and return the structured intent ---
    # If any essential component is missing, the intent is invalid
    if not all([metric, dimension, chart_type]):
        return None

    return {"metric": metric, "dimension": dimension, "chart_type": chart_type}


log_success("parse_query defined — template-matching intent classifier.")

In [ ]:
# =============================================================================
# Section 2: Demo — Intent Classification on Sample Queries
# Ref: Section 7.2, p. 183 — "For a query like 'What was the trend of clicks
#       over time?', it identifies 'clicks' as the metric..."
# =============================================================================

log_info("=" * 60)
log_info("Section 2 Demo: Intent Classification")
log_info("=" * 60)

demo_queries = [
    "What was the trend of clicks over time?",
    "Show me spend by campaign",
    "Which campaign had the most conversions?",
    "Show me conversions over time",
    "Tell me something interesting",          # Should fail — no metric/dimension
    "What about clicks?",                     # Should fail — no dimension
]

for q in demo_queries:
    intent = parse_query(q)
    if intent:
        log_info(f"Query:  '{q}'")
        log_success(
            f"  → metric={intent['metric']}, "
            f"dimension={intent['dimension']}, "
            f"chart_type={intent['chart_type']}"
        )
    else:
        log_info(f"Query:  '{q}'")
        log_warning("  → Could not parse intent (missing metric or dimension).")
    print()

> **📘 Implementation Insight: Failure Memory as Circuit Breaker** *(Book p. 184)*
>
> To prevent an agent from repeatedly attempting to use a tool that is clearly failing, the system can implement a short-term **failure memory**. This acts like a circuit breaker: if a tool fails multiple times in a short window, the agent temporarily marks it as unavailable and does not attempt to select it again for a predefined period, preventing wasted cycles.
>
> Production-ready agents implement a layered defense: **safe invocation wrappers** (try/except with retry logic), **fallback tool chains** (alternative tools for the same goal), **confidence-based switching** (discard low-confidence results and retry with a different tool), and **escalation paths** to human operators as the final fallback.


---
## Section 3: Error Handling and Resilience

**Chapter Reference:** Section 7.3 — Error Handling in Tool Integration

Production-ready agents implement a layered defense against four categories of failure:

1. **Input validation errors** — Data doesn't match the expected schema
2. **Runtime failures** — Network timeouts, API errors, file I/O exceptions
3. **Semantic mismatches** — Tool succeeds but produces the wrong result
4. **Tool unavailability** — External service is offline or deprecated

This section builds the `data_viz_agent` orchestrator, which ties together intent parsing,
tool selection, and error handling into a single **Think → Plan → Act** cycle.
It then demonstrates deliberate failure scenarios to show the resilience layer in action.

In [ ]:
# =============================================================================
# Section 3: data_viz_agent — The Central Orchestrator
# Ref: Section 7.3, pp. 185–186 — "Implementation Example"
#
# This function implements the full Think/Plan/Act cycle:
#   THINK: parse_query() interprets the user's goal
#   PLAN:  Dynamically construct a list of tool calls based on intent
#   ACT:   Execute tools sequentially, passing data_state between steps
# =============================================================================

def data_viz_agent(query: str, file_path: str) -> Optional[str]:
    """Central orchestrator for the data visualization assistant.
    Implements the Think/Plan/Act cycle from Section 7.1."""

    log_info("=" * 60)
    log_step("7.3", 0, f"data_viz_agent received query: '{query}'")
    log_info("=" * 60)

    # ----- THINK: Interpret the user's goal -----
    log_step("7.3", 1, "THINK — Parsing user intent...")
    intent = parse_query(query)
    if intent is None:
        log_error("Could not understand the request. "
                  "Query must contain a metric (spend/clicks/conversions) "
                  "AND a dimension (by campaign / over time / trend).")
        return None

    log_info(f"  Intent Analysis: metric={intent['metric']}, "
             f"dimension={intent['dimension']}, chart_type={intent['chart_type']}")

    # ----- PLAN: Create a sequence of tool calls -----
    log_step("7.3", 2, "PLAN — Constructing execution plan...")
    plan = [
        ("load_csv", {"file_path": file_path}),
        ("group_by_and_aggregate", {
            "group_by_col": intent["dimension"],
            "agg_col": intent["metric"],
            "agg_func": "sum",
        }),
    ]

    if intent["chart_type"] == "bar":
        plot_tool = "plot_bar_chart"
        base_title = f"{intent['metric']} by {intent['dimension']}"
    else:
        plot_tool = "plot_line_chart"
        base_title = f"{intent['metric']} over {intent['dimension']}"

    output_path = f"outputs/output_{intent['metric']}_{intent['dimension']}.png"
    plan.append((plot_tool, {
        "x_col": intent["dimension"],
        "y_col": intent["metric"],
        "title": base_title,
        "output_path": output_path,
    }))

    log_info("  Action Plan:")
    for step_name, _ in plan:
        log_info(f"    - {step_name}")

    # ----- ACT: Execute the plan step-by-step -----
    log_step("7.3", 3, "ACT — Executing plan...")
    data_state = None

    tool_functions = {
        "load_csv": load_csv,
        "group_by_and_aggregate": group_by_and_aggregate,
        "plot_bar_chart": plot_bar_chart,
        "plot_line_chart": plot_line_chart,
    }

    for tool_name, tool_args in plan:
        tool_func = tool_functions[tool_name]

        # Inject data_state as the first argument for downstream tools
        if tool_name != "load_csv":
            tool_args["df"] = data_state

        result = tool_func(**tool_args)

        if isinstance(result, pd.DataFrame):
            if result.empty:
                log_error(f"Tool '{tool_name}' returned empty DataFrame. Aborting plan.")
                return None
            data_state = result
        elif result is None and tool_name != "load_csv":
            log_error(f"Execution failed at '{tool_name}'. Aborting plan.")
            return None

    log_success(f"data_viz_agent completed. Output: {output_path}")
    return output_path

In [ ]:
# =============================================================================
# Section 3: Demo — Successful Orchestration
# Ref: Section 7.3 — data_viz_agent with valid queries
# =============================================================================

log_info("=" * 60)
log_info("Section 3 Demo: Successful Orchestration Runs")
log_info("=" * 60)

# Query 1: Bar chart of spend by campaign
data_viz_agent("Show me spend by campaign", "data/sample_ad_campaign.csv")
print()

# Query 2: Line chart of clicks over time
data_viz_agent("What was the trend of clicks over time?", "data/sample_ad_campaign.csv")

In [ ]:
# =============================================================================
# Section 3: Deliberate Failure Demo 1 — FileNotFoundError
# Ref: Section 7.3, Strategy §6.6 — "Force FileNotFoundError"
#
# Expected behavior:
#   - load_csv catches FileNotFoundError
#   - Red [ERROR] log appears
#   - @graceful_fallback returns empty DataFrame
#   - Orchestrator detects df.empty and aborts cleanly
# =============================================================================

log_info("=" * 60)
log_warning("Section 3: DELIBERATE FAILURE DEMO — FileNotFoundError")
log_info("=" * 60)

result = data_viz_agent(
    "Show me spend by campaign",
    "nonexistent_file.csv",  # This file does not exist
)

if result is None:
    log_info("Orchestrator handled the failure gracefully. No crash.")

In [ ]:
# =============================================================================
# Section 3: Deliberate Failure Demo 2 — Column Mismatch
# Ref: Section 7.3, Strategy §6.6 — "Force Column Mismatch"
#
# Expected behavior:
#   - group_by_and_aggregate detects invalid column "bad_column"
#   - Raises ValueError caught by @graceful_fallback
#   - Red [ERROR] log, returns None
#   - Orchestrator detects None and aborts cleanly
# =============================================================================

log_info("=" * 60)
log_warning("Section 3: DELIBERATE FAILURE DEMO — Column Mismatch")
log_info("=" * 60)

# First load a valid dataframe, then force a bad column name
df_valid = load_csv("data/sample_ad_campaign.csv")
if not df_valid.empty:
    bad_result = group_by_and_aggregate(df_valid, "bad_column", "spend", "sum")
    if bad_result is None:
        log_info("group_by_and_aggregate returned None for invalid column. "
                 "Orchestrator would abort cleanly at this point.")

log_success("Section 3 complete. Both failure demos handled gracefully.")

---
## Section 4: The Chain-of-Agents Orchestrator

**Chapter Reference:** Sections 7.4–7.5

Complex tasks often exceed the capacity of any single agent. A chain-of-agents orchestrator
coordinates a team of specialists, each with its own domain expertise and tool chest.

A robust **Cooperation Protocol** is built upon four key architectural pillars (Table 7.1):

| Protocol Element | Definition |
|:---|:---|
| **Message Format** | A shared envelope structure (e.g., JSON with sender, recipient, task_id, payload) |
| **Role Declaration** | Each agent registers its name, capabilities, and accepted task types |
| **Task Delegation Scheme** | A typed request specifying target agent, action, inputs, and expected output schema |
| **Status Signaling** | Standardized status field (pending, running, done, error) for sequencing |

This section implements a **Market Intelligence System** with three specialist agents that
contribute findings to shared **episodic memory**, coordinated by a manager agent.

> **📘 Standards Connection: MCP and A2A Protocols** *(Book p. 188)*
>
> The cooperation protocol elements (message format, role declaration, task delegation scheme, status signaling) align with emerging industry standards. The **Model Context Protocol (MCP)** and **Agent-to-Agent (A2A) protocol**, introduced in Chapter 6, formalize exactly this message schema and role-declaration layer into open, interoperable specifications.
>
> Where Chapter 6 covered their structure and negotiation mechanics, this chapter applies those foundations: the cooperation protocol described here is the practical expression of what MCP and A2A standardize at the wire level.


#### Figure 7.3 — Memory-Augmented Agent Architecture *(Book p. 189)*

```
  External Inputs ──▶ Perception ──▶ Agent Core (Cognition & Reasoning)
                                         │              │
                               Active Context      Memory Query
                               Manager   │              │
                                  ▼      ▼              ▼
                        ┌─────────────┐  ┌──────────────────────┐
                        │   Working    │  │  Vector Databases    │◀── External
                        │   Memory     │  │  & Retrieval         │    DBs/APIs
                        │ (Current     │  └──────────┬───────────┘
                        │  Prompt)     │             │
                        └──────┬───────┘    ┌────────┼────────┐
                               │           ▼                 ▼
                    Store Session    ┌──────────────┐  ┌──────────────┐
                    Context/Events   │  Episodic    │  │  Semantic    │
                               │    │  Memory      │  │  Memory      │
                               └───▶│ (Historical  │  │ (Facts &     │
                                    │  Interactions)│  │  Knowledge)  │◀── Ingestion
                                    └──────────────┘  └──────────────┘
```

**Working memory** holds the immediate context (scratchpad). **Episodic memory** stores historical interaction logs. **Semantic memory** contains durable factual knowledge. The Agent Core queries long-term memory and loads relevant context into working memory for each task.


In [ ]:
# =============================================================================
# Section 4: Specialist Agents — NewsAgent, FinancialAgent, SentimentAgent
# Ref: Section 7.5, pp. 190–191 — "Implementation Example: Market Intelligence"
#
# Each specialist accepts a company name and returns a structured payload.
# All three wrap results in the same dict layout:
#   {"source": ..., "status": ..., "data": ...}
# making outputs predictable for the orchestrator.
#
# Mock data is drawn directly from the chapter's own code listings.
# In production, uncomment the API calls shown in comments.
# =============================================================================

@graceful_fallback(
    fallback_return={"source": "NewsAgent", "status": "error", "data": []},
    section="7.5",
)
def NewsAgent(company_name: str) -> Dict[str, Any]:
    """Fetches top news headlines from a data source."""
    log_step("7.5", 1, f"[NewsAgent] Fetching headlines for '{company_name}'...")
    headlines = [
        f"{company_name}: Q3 earnings beat analyst estimates by 8%",
        f"{company_name} announces expansion into cloud infrastructure",
        f"Regulatory review concluded; {company_name} cleared to proceed",
    ]
    # In production, uncomment: response = news_api.get(company_name)
    response = {"data": headlines}
    return {"source": "NewsAgent", "status": "success", "data": response["data"]}


@graceful_fallback(
    fallback_return={"source": "FinancialAgent", "status": "error", "data": {}},
    section="7.5",
)
def FinancialAgent(company_name: str) -> Dict[str, Any]:
    """Fetches financial data from a data source."""
    log_step("7.5", 2, f"[FinancialAgent] Fetching financial data for '{company_name}'...")
    response = {
        "data": {
            "pe_ratio": 24.5,
            "revenue_growth": 0.12,
            "debt_to_equity": 0.38,
            "last_close": 142.73,
        }
    }
    # In production, uncomment: response = financial_api.get(company_name)
    return {"source": "FinancialAgent", "status": "success", "data": response["data"]}


@graceful_fallback(
    fallback_return={"source": "SentimentAgent", "status": "error", "data": {}},
    section="7.5",
)
def SentimentAgent(company_name: str) -> Dict[str, Any]:
    """Fetches a sentiment score from a data source."""
    log_step("7.5", 3, f"[SentimentAgent] Fetching sentiment for '{company_name}'...")
    response = {
        "data": {
            "score": 0.72,
            "label": "positive",
            "evidence": [
                f"Investor confidence in {company_name} remains high.",
                "Social media tone broadly favorable this week.",
            ],
        }
    }
    # In production, uncomment: response = sentiment_api.get(company_name)
    return {"source": "SentimentAgent", "status": "success", "data": response["data"]}


log_success("Specialist agents defined: NewsAgent, FinancialAgent, SentimentAgent.")

In [ ]:
# =============================================================================
# Section 4: Manager Agent with Episodic Memory
# Ref: Section 7.4, pp. 187–189 — Cooperation Protocol
# Ref: Section 7.5, pp. 189–193 — Memory-Augmented Multi-Agent Systems
#
# The manager agent coordinates the specialists, stores results in episodic
# memory (timestamped interaction log), and prepares context for synthesis.
# =============================================================================

from datetime import datetime


class ManagerAgent:
    """Orchestrates specialist agents and maintains shared episodic memory.

    Implements the Cooperation Protocol from Section 7.4:
    - Role Declaration: each agent has a defined role and output schema
    - Task Delegation: manager dispatches to each specialist in turn
    - Status Signaling: checks 'status' field in agent responses
    - Shared Memory: episodic_memory stores timestamped interaction records
    """

    def __init__(self, company_name: str):
        self.company_name = company_name
        self.episodic_memory: List[Dict[str, Any]] = []  # Section 7.5 — episodic memory
        self.results: Dict[str, Any] = {}

    def delegate_and_collect(self) -> Dict[str, Any]:
        """Dispatch tasks to each specialist and store results in episodic memory."""
        log_info("=" * 60)
        log_step("7.4", 1, f"ManagerAgent dispatching tasks for '{self.company_name}'...")
        log_info("=" * 60)

        specialists = {
            "NewsAgent": NewsAgent,
            "FinancialAgent": FinancialAgent,
            "SentimentAgent": SentimentAgent,
        }

        for agent_name, agent_func in specialists.items():
            result = agent_func(self.company_name)
            self.results[agent_name] = result

            # Write to episodic memory — Section 7.5
            memory_entry = {
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "agent": agent_name,
                "status": result.get("status", "unknown"),
                "data_summary": str(result.get("data", ""))[:120],
            }
            self.episodic_memory.append(memory_entry)
            log_info(f"  Episodic memory updated: {agent_name} -> {result['status']}")

        log_success(f"All specialists reported. {len(self.episodic_memory)} memory entries.")
        return self.results

    def show_episodic_memory(self):
        """Display the episodic memory log for inspection."""
        log_info("Episodic Memory Log:")
        for entry in self.episodic_memory:
            log_info(
                f"  [{entry['timestamp']}] {entry['agent']}: "
                f"status={entry['status']}, "
                f"data={entry['data_summary'][:80]}..."
            )


log_success("ManagerAgent class defined with episodic memory.")

In [ ]:
# =============================================================================
# Section 4: Demo — Market Intelligence System
# Ref: Sections 7.4-7.5 — Manager delegates to 3 specialists
# =============================================================================

log_info("=" * 60)
log_info("Section 4 Demo: Market Intelligence System for 'TechCorp'")
log_info("=" * 60)

manager = ManagerAgent("TechCorp")
results = manager.delegate_and_collect()

print()
manager.show_episodic_memory()

print()
log_info("Raw specialist results:")
for agent_name, result in results.items():
    print(f"  {agent_name}: {json.dumps(result, indent=2, default=str)}")

---
## Section 5: Conflict Resolution Mechanisms

**Chapter Reference:** Section 7.6

When multiple specialists analyze complex information, disagreements are inevitable.
The architectural strength of a multi-agent system is measured by its capacity to
**resolve conflict productively** (Figure 7.4).

The arbitration workflow consists of four stages:
1. **Conflict Detection** — Semantic similarity or numerical divergence check
2. **Automated Arbitration** — Arbiter agent consults a knowledge base
3. **Confidence-Based Consensus** — Accept if confidence exceeds threshold (e.g., 95%)
4. **Human Escalation** — Route to human reviewer if confidence is low

This section extends the `ManagerAgent` with a `synthesize_report` method that computes
a `conflict_score` to detect divergence between sentiment and financial signals.

In [ ]:
# =============================================================================
# Section 5: Conflict Resolution — synthesize_report with conflict_score
# Ref: Section 7.6, pp. 193–194 — "Implementation Example"
#
# conflict_score = abs(sentiment_score - (stock_change / 10))
#
# sentiment_score is bounded [-1, 1] by the SentimentAgent.
# stock_change is expressed as a percentage (e.g., 5.0 for 5%).
# Dividing by 10 maps a typical daily swing of +/-10% onto the same [-1, 1] scale.
# The 0.5 threshold represents a half-unit divergence on this normalized scale.
# =============================================================================

def synthesize_report(
    results: Dict[str, Any],
    stock_change_pct: float = 5.0,
) -> str:
    """Aggregate specialist findings into a single report with conflict detection.

    Parameters
    ----------
    results : dict
        Output from ManagerAgent.delegate_and_collect().
    stock_change_pct : float
        Simulated stock price change percentage for conflict detection.
        Default 5.0 represents a 5% positive move — aligned with chapter example.
    """
    log_step("7.6", 1, "Synthesizing market intelligence report...")

    financial_data = results.get("FinancialAgent", {}).get("data", {})
    sentiment_data = results.get("SentimentAgent", {}).get("data", {})
    news_items = results.get("NewsAgent", {}).get("data", [])

    # Extract key values
    sentiment_score = sentiment_data.get("score", 0.0)

    # Conflict score formula from Section 7.6, p. 27
    # Normalizes both inputs to a compatible scale before comparing
    conflict_score = abs(sentiment_score - (stock_change_pct / 10))

    # Build the report
    headlines = "; ".join(news_items[:2]) if news_items else "no headlines"

    report = "Market Intelligence Report\n"
    report += "=" * 40 + "\n"
    report += f"Recent news: {headlines}\n"
    report += (
        f"P/E ratio: {financial_data.get('pe_ratio', 'N/A')}, "
        f"Revenue growth: {financial_data.get('revenue_growth', 0):.0%}\n"
    )

    # Conflict detection — threshold 0.5 (Section 7.6, p. 27)
    if conflict_score > 0.5:
        report += (
            f"\n**Conflict Detected (Score: {conflict_score:.2f}):** "
            f"A significant discrepancy exists between public sentiment "
            f"(score={sentiment_score}) and market performance "
            f"(change={stock_change_pct}%). Further investigation recommended."
        )
        log_warning(
            f"Conflict detected! Score: {conflict_score:.2f} > 0.5 threshold. "
            f"Sentiment={sentiment_score}, StockChange={stock_change_pct}%"
        )
    else:
        report += (
            f"\n**Alignment Confirmed (Score: {conflict_score:.2f}):** "
            f"Public perception and market performance are well-aligned. "
            f"No material discrepancy detected."
        )
        log_success(
            f"Alignment confirmed. Conflict score: {conflict_score:.2f} "
            f"(within 0.5 threshold)."
        )

    return report


log_success("synthesize_report function defined with conflict_score detection.")

In [ ]:
# =============================================================================
# Section 5: Demo — Conflict Resolution in Action
# Ref: Section 7.6 — Two scenarios: aligned signals vs. conflicting signals
# =============================================================================

log_info("=" * 60)
log_info("Section 5 Demo: Conflict Resolution")
log_info("=" * 60)

# --- Scenario A: Aligned signals ---
# sentiment_score=0.72, stock_change=5.0% -> conflict_score = |0.72 - 0.5| = 0.22
log_info("Scenario A: Aligned signals (stock +5%, sentiment 0.72)")
report_aligned = synthesize_report(results, stock_change_pct=5.0)
print()
print(report_aligned)
print()

# --- Scenario B: Conflicting signals ---
# sentiment_score=0.72, stock_change=-8.0% -> conflict_score = |0.72 - (-0.8)| = 1.52
log_info("Scenario B: Conflicting signals (stock -8%, sentiment 0.72)")
report_conflict = synthesize_report(results, stock_change_pct=-8.0)
print()
print(report_conflict)

log_success("Section 5 complete. Both conflict scenarios demonstrated.")

---
## Section 6: Agentic Workflow — E-Commerce Order Processing

**Chapter Reference:** Section 7.7

An agentic workflow system extends orchestration into **long-running, stateful business
processes**. Unlike short-lived tool invocations, workflows require persistence, branching,
error recovery, and checkpoints for human oversight.

This section builds an e-commerce order processing workflow with:
- **Deterministic tools:** `validate_order`, `check_inventory`, `process_payment`
- **Intelligent agent:** `llm_analyst_agent` for risk assessment (with rule-based fallback)
- **Human-in-the-Loop (HITL) gate:** Pauses for human judgment on medium/high risk orders

Three test orders demonstrate the full range of behaviors:

| Order ID | Total | Risk Trigger | Expected Outcome |
|:---|:---|:---|:---|
| ORD-1001 | $69.98 | None (standard) | Happy path — auto-completes |
| ORD-1002 | $1,599.70 | High value + new customer | HITL escalation (medium risk) |
| ORD-1003 | $2,399.40 | Address mismatch + high value | HITL escalation (high risk) |

In [ ]:
# =============================================================================
# Section 6: E-Commerce Workflow Tools
# Ref: Section 7.7, pp. 195–197 — Case Study: E-Commerce Order Processing
#
# Inventory database: a small DataFrame simulating product stock.
# Each tool is wrapped with @graceful_fallback for resilience.
# =============================================================================

# --- Inventory Database (simulated) ---
inventory_db = pd.DataFrame({
    "product_id": ["PROD-A", "PROD-B", "PROD-C", "PROD-D"],
    "product_name": ["Wireless Mouse", "USB-C Hub", "Mechanical Keyboard", "Monitor Stand"],
    "price": [34.99, 49.99, 89.99, 59.99],
    "stock": [150, 75, 40, 20],
})

log_info("Inventory database initialized:")
print(inventory_db.to_string(index=False))
print()

# --- Test Orders (Strategy §6.3) ---
test_orders = [
    {
        "order_id": "ORD-1001",
        "customer_id": "CUST-200",
        "items": [{"product_id": "PROD-A", "quantity": 2}],
        "shipping_address": "123 Main St, Springfield",
        "billing_address": "123 Main St, Springfield",
        "customer_type": "returning",
    },
    {
        "order_id": "ORD-1002",
        "customer_id": "CUST-501",
        "items": [
            {"product_id": "PROD-C", "quantity": 10},
            {"product_id": "PROD-D", "quantity": 10},
        ],
        "shipping_address": "456 Oak Ave, Shelbyville",
        "billing_address": "456 Oak Ave, Shelbyville",
        "customer_type": "new",
    },
    {
        "order_id": "ORD-1003",
        "customer_id": "CUST-777",
        "items": [
            {"product_id": "PROD-C", "quantity": 20},
            {"product_id": "PROD-D", "quantity": 8},
        ],
        "shipping_address": "789 Elm Rd, Capital City",
        "billing_address": "999 Different Blvd, Ogdenville",
        "customer_type": "new",
    },
]

log_info(f"{len(test_orders)} test orders prepared.")

In [ ]:
# =============================================================================
# Section 6: Workflow Step Functions
# Ref: Section 7.7, pp. 195–197
# =============================================================================

@graceful_fallback(fallback_return=(False, 0.0), section="7.7")
def validate_order(order: Dict[str, Any], inv_db: pd.DataFrame) -> Tuple[bool, float]:
    """Step 1: Validate order data and compute total."""
    order_id = order.get("order_id", "N/A")
    log_step("7.7", 1, f"Validating Order #{order_id}...")

    total = 0.0
    for item in order.get("items", []):
        product = inv_db[inv_db["product_id"] == item["product_id"]]
        if product.empty:
            raise ValueError(f"Product {item['product_id']} not found in inventory.")
        total += product.iloc[0]["price"] * item["quantity"]

    log_info(f"  Order #{order_id} validated. Total: ${total:,.2f}")
    return (True, total)


@graceful_fallback(fallback_return=False, section="7.7")
def check_inventory(order: Dict[str, Any], inv_db: pd.DataFrame) -> bool:
    """Step 3: Check and reserve inventory for all items."""
    order_id = order.get("order_id", "N/A")
    log_step("7.7", 3, f"Checking inventory for Order #{order_id}...")

    for item in order.get("items", []):
        product = inv_db[inv_db["product_id"] == item["product_id"]]
        if product.empty or product.iloc[0]["stock"] < item["quantity"]:
            log_error(f"  Insufficient stock for {item['product_id']}.")
            return False
    log_info(f"  Inventory confirmed for Order #{order_id}.")
    return True


@graceful_fallback(fallback_return=False, section="7.7")
def process_payment(order: Dict[str, Any], order_total: float) -> bool:
    """Step 4: Process payment (simulated)."""
    order_id = order.get("order_id", "N/A")
    log_step("7.7", 4, f"Processing payment of ${order_total:,.2f} for Order #{order_id}...")
    # Simulate payment processing delay
    time.sleep(0.5)
    log_success(f"  Payment of ${order_total:,.2f} processed for Order #{order_id}.")
    return True


log_success("Workflow step functions defined: validate_order, check_inventory, process_payment.")

In [ ]:
# =============================================================================
# Section 6: LLM Analyst Agent — Risk Assessment
# Ref: Section 7.7, p. 196 — "Embedding Intelligence in Workflow Nodes"
#
# The llm_analyst_agent is designed to be adaptable:
# - Live Mode: uses the OpenAI API for nuanced risk analysis
# - Simulation Mode: uses rule-based logic as a mock analysis
#
# Rule-based risk triggers (from chapter text):
# - High value (> $500) + new customer → medium risk
# - Address mismatch + high value → high risk
# =============================================================================

@graceful_fallback(
    fallback_return={"risk_level": "high", "reason": "Unavailable — default high."},
    section="7.7",
)
def llm_analyst_agent(order: Dict[str, Any], order_total: float) -> Dict[str, str]:
    """Step 2: Assess risk for an order using LLM or rule-based fallback."""
    order_id = order.get("order_id", "N/A")
    log_step("7.7", 2, f"Assessing risk for Order #{order_id}...")

    if not LIVE_MODE:
        # --- Rule-based mock analysis (Simulation Mode) ---
        log_mock("  Using mock analysis (Simulation Mode).")

        address_mismatch = (
            order.get("shipping_address", "") != order.get("billing_address", "")
        )
        is_new = order.get("customer_type", "") == "new"
        high_value = order_total > 500.0

        if address_mismatch and high_value:
            analysis = {
                "risk_level": "high",
                "reason": "Shipping/billing address mismatch combined with high order value.",
            }
        elif high_value and is_new:
            analysis = {
                "risk_level": "medium",
                "reason": "Order total exceeds typical range for a new customer.",
            }
        else:
            analysis = {
                "risk_level": "low",
                "reason": "Standard order parameters. No risk indicators detected.",
            }

        log_info(f"  Risk Level: {analysis['risk_level']}")
        log_info(f"  Reason: {analysis['reason']}")
        return analysis

    # --- Live Mode: use OpenAI API ---
    log_info("  Mode: Using OpenAI GPT for analysis.")
    try:
        prompt = (
            f"Assess the fraud risk for Order #{order_id}. "
            f"Total: ${order_total:,.2f}. Customer type: {order.get('customer_type')}. "
            f"Shipping: {order.get('shipping_address')}. "
            f"Billing: {order.get('billing_address')}. "
            f"Respond with JSON: {{\"risk_level\": \"low|medium|high\", \"reason\": \"...\"}}."
        )
        response_text = llm.generate(prompt)
        analysis = json.loads(response_text)
        log_info(f"  LLM Analysis complete. Risk Level: {analysis.get('risk_level')}")
        return analysis
    except Exception as e:
        log_error(f"  LLM analysis failed: {e}. Defaulting to high risk.")
        return {"risk_level": "high", "reason": "LLM analysis failed."}


log_success("llm_analyst_agent defined with Live/Simulation dual mode.")

In [ ]:
# =============================================================================
# Section 6: Workflow Manager Agent with HITL Gate
# Ref: Section 7.7, pp. 195–197 — workflow_manager_agent
# Ref: Strategy §6.5 — HITL behavior in Simulation Mode
#
# Step 2.5: HITL Escalation Gate
#   If risk_level is medium or high, pause for human judgment.
#   In Simulation Mode: auto-approve after 2s delay.
#   In Live Mode + INTERACTIVE_MODE: use real input() prompt.
# =============================================================================

def workflow_manager_agent(order: Dict[str, Any], inv_db: pd.DataFrame) -> bool:
    """Orchestrate the full e-commerce order processing workflow."""
    order_id = order.get("order_id", "N/A")
    log_info("=" * 60)
    log_info(f"Starting Workflow for Order #{order_id}")
    log_info("=" * 60)

    try:
        # Step 1: Validate the order data
        is_valid, order_total = validate_order(order, inv_db)
        if not is_valid:
            raise AgentError("Order validation failed.")

        # Step 2: Assess fraud risk with an intelligent agent
        risk_analysis = llm_analyst_agent(order, order_total)
        risk_level = risk_analysis.get("risk_level", "high").lower()
        risk_reason = risk_analysis.get("reason", "No reason provided.")

        # Step 2.5: Human-in-the-Loop Escalation Gate
        if risk_level in ["high", "medium"]:
            log_warning(f"HUMAN INTERVENTION REQUIRED for Order #{order_id}!")
            log_info(f"  Reason: Analyst flagged order as '{risk_level}' — {risk_reason}")

            if LIVE_MODE and INTERACTIVE_MODE:
                # Real interactive prompt
                decision = ""
                while decision not in ["approve", "reject"]:
                    decision = input("  Type 'approve' or 'reject': ").lower().strip()
            else:
                # Simulation Mode: auto-approve after delay (Strategy §6.5)
                log_mock("  Auto-approving after 2s delay (Simulation Mode)...")
                time.sleep(2)
                decision = "approve"

            if decision == "reject":
                log_error(f"  DECISION: Human operator REJECTED Order #{order_id}.")
                raise AgentError("Workflow terminated by human operator.")
            else:
                log_success(f"  DECISION: Human operator APPROVED Order #{order_id}. Resuming...")
        else:
            log_success(f"  Risk level 'low' — no HITL required for Order #{order_id}.")

        # Step 3: Check and reserve inventory
        if not check_inventory(order, inv_db):
            raise AgentError("Inventory check failed.")

        # Step 4: Process the payment
        if not process_payment(order, order_total):
            raise AgentError("Payment processing failed.")

        log_success(f"Workflow for Order #{order_id} COMPLETED SUCCESSFULLY.")
        return True

    except AgentError as e:
        log_error(f"Workflow for Order #{order_id} TERMINATED. Reason: {e}")
        return False


log_success("workflow_manager_agent defined with HITL escalation gate.")

In [ ]:
# =============================================================================
# Section 6: Demo — Run All 3 Test Orders
# Ref: Strategy §6.3 — Expected outcomes:
#   ORD-1001: Happy path (low risk, auto-completes)
#   ORD-1002: Medium risk HITL escalation (auto-approved in Simulation Mode)
#   ORD-1003: High risk HITL escalation (auto-approved in Simulation Mode)
# =============================================================================

log_info("=" * 60)
log_info("Section 6 Demo: E-Commerce Workflow — 3 Test Orders")
log_info("=" * 60)

workflow_results = {}
for order in test_orders:
    success = workflow_manager_agent(order, inventory_db)
    workflow_results[order["order_id"]] = success
    print()

# Summary table
log_info("=" * 60)
log_info("Workflow Results Summary:")
for oid, result in workflow_results.items():
    status = "COMPLETED" if result else "TERMINATED"
    badge_func = log_success if result else log_error
    badge_func(f"  {oid}: {status}")

log_success("Section 6 complete. All 3 test orders processed.")

---
## Section 7: Agentic Workflow — Insurance Claims Processing

**Chapter Reference:** Section 7.7b

This section demonstrates a more complex, multi-agent workflow modeled as a **state machine**
(Figure 7.5). Five specialized agents handle different stages of insurance claims processing:

| Agent | Role |
|:---|:---|
| **Intake Agent** | OCR/NLP to digitize claim forms |
| **Validator Agent** | Checks policy status, fraud signals, required documentation |
| **Classifier Agent** | Assesses claim type, urgency, and risk level |
| **Payout Agent** | Calculates settlement and processes payment |
| **Escalation Agent** | Routes high-risk or low-confidence claims to human review |

**Guard Conditions** (Table 7.2):

| Guard | Action |
|:---|:---|
| `claim_amount > threshold` | Route to human for approval |
| Validation fails | Transition to Closed: Rejected |
| `confidence_score < 0.85` | Escalate to Pending Human Review |

Three test claims:

| Claim ID | Amount | Type | Expected Path |
|:---|:---|:---|:---|
| CLM-4821 | $8,400 | water_damage | Auto-approved (confidence 0.91) |
| CLM-5099 | $47,000 | fire_damage | Escalated (confidence 0.79) |
| CLM-5100 | $3,200 | theft | Rejected at validation (expired policy) |

#### Figure 7.5 — Insurance Claim State Machine *(Book p. 199)*

```
  ● ──▶ [Intake] ──Form Submitted──▶ [Validating] ──Valid──▶ [Assessing Risk]
                                         │                        │         │
                                      Invalid               Low Risk   High Risk
                                         │                        │         │
                                         ▼                        ▼         ▼
                                  [Closed:               [Processing  [Pending Human
                                   Rejected]              Payout]      Review]
                                      ▲                  │    │          │      │
                                      │            Success│ Payment  Approved  Rejected
                                      │                  │  Failed     │      │
                                      │                  ▼    │        ▼      │
                            Rejected  │           [Closed:    └──▶[Processing │
                            (Human)───┘            Approved]       Payout]    │
                                  ▲                                           │
                                  └───────────────────────────────────────────┘
```

**Guard conditions** (Table 7.2): `claim_amount > threshold` → human approval; `validation fails` → rejected; `confidence_score < 0.85` → escalate to human review. At each transition, the system records the agent's reasoning, tools used, and human decisions for a complete audit trail.


In [ ]:
# =============================================================================
# Section 7: Insurance Claims — Test Data and Policy Database
# Ref: Section 7.7b, pp. 198–200 — Multi-Agent Insurance Claims Workflow
# Ref: Strategy §6.4 — Test claims
# =============================================================================

# --- Policy Database (simulated) ---
policy_db = {
    "POL-992317": {"status": "active", "holder": "Alice Johnson", "fraud_flag": False},
    "POL-110482": {"status": "active", "holder": "Bob Martinez", "fraud_flag": False},
    "POL-EXPIRED": {"status": "expired", "holder": "Charlie Lee", "fraud_flag": False},
}

# --- Test Claims (Strategy §6.4) ---
test_claims = [
    {
        "claim_id": "CLM-4821",
        "policy_id": "POL-992317",
        "claim_type": "water_damage",
        "amount": 8400.00,
        "description": "Burst pipe caused flooding in basement. Damage to walls and flooring.",
    },
    {
        "claim_id": "CLM-5099",
        "policy_id": "POL-110482",
        "claim_type": "fire_damage",
        "amount": 47000.00,
        "description": "Kitchen fire spread to living area. Significant structural damage.",
    },
    {
        "claim_id": "CLM-5100",
        "policy_id": "POL-EXPIRED",
        "claim_type": "theft",
        "amount": 3200.00,
        "description": "Laptop and electronics stolen during home break-in.",
    },
]

log_info(f"Policy database: {len(policy_db)} policies loaded.")
log_info(f"Test claims: {len(test_claims)} claims prepared.")

In [ ]:
# =============================================================================
# Section 7: Insurance Claims — Five Specialized Agents
# Ref: Section 7.7b, pp. 198–199 — Agent roles
# =============================================================================

@graceful_fallback(fallback_return=None, section="7.7b")
def intake_agent(claim: Dict[str, Any]) -> Dict[str, Any]:
    """Intake Agent: Digitize and extract fields from claim submission.
    In production, this would use OCR and NLP on submitted forms."""
    claim_id = claim.get("claim_id", "N/A")
    log_step("7.7b", 1, f"[Intake] Processing claim {claim_id}...")

    # Simulate OCR/NLP field extraction
    extracted = {
        "claim_id": claim["claim_id"],
        "policy_id": claim["policy_id"],
        "claim_type": claim["claim_type"],
        "amount": claim["amount"],
        "description": claim["description"],
        "status": "intake_complete",
    }
    log_info(f"  Extracted: type={extracted['claim_type']}, amount=${extracted['amount']:,.2f}")
    return extracted


@graceful_fallback(fallback_return=False, section="7.7b")
def validator_agent(claim_record: Dict[str, Any], policies: Dict) -> bool:
    """Validator Agent: Check policy status and fraud signals.
    Guard: validation fails -> Closed: Rejected."""
    claim_id = claim_record.get("claim_id", "N/A")
    policy_id = claim_record.get("policy_id", "N/A")
    log_step("7.7b", 2, f"[Validator] Validating claim {claim_id}, policy {policy_id}...")

    policy = policies.get(policy_id)
    if policy is None:
        log_error(f"  Policy {policy_id} not found in database.")
        return False

    if policy["status"] != "active":
        log_error(f"  Policy {policy_id} status: {policy['status']}. Claim rejected.")
        return False

    if policy.get("fraud_flag", False):
        log_warning(f"  Fraud flag detected on policy {policy_id}.")
        return False

    log_info(f"  Policy {policy_id} validated: active, no fraud flag.")
    return True


@graceful_fallback(
    fallback_return={"confidence_score": 0.5, "risk": "high"},
    section="7.7b",
)
def classifier_agent(claim_record: Dict[str, Any]) -> Dict[str, Any]:
    """Classifier Agent: Assess claim type, urgency, and risk level.
    Guard: confidence_score < 0.85 -> escalate to Pending Human Review."""
    claim_id = claim_record.get("claim_id", "N/A")
    claim_type = claim_record.get("claim_type", "unknown")
    amount = claim_record.get("amount", 0.0)
    log_step("7.7b", 3, f"[Classifier] Assessing risk for claim {claim_id}...")

    # Rule-based classification (Simulation Mode)
    # High-value claims or fire/explosion types get lower confidence
    if amount > 25000 or claim_type in ["fire_damage", "explosion"]:
        classification = {
            "confidence_score": 0.79,
            "risk": "high",
            "claim_type": claim_type,
        }
    else:
        classification = {
            "confidence_score": 0.91,
            "risk": "low",
            "claim_type": claim_type,
        }

    log_info(
        f"  Classification: confidence={classification['confidence_score']}, "
        f"risk={classification['risk']}"
    )
    return classification


@graceful_fallback(fallback_return=False, section="7.7b")
def payout_agent(claim_record: Dict[str, Any]) -> bool:
    """Payout Agent: Calculate settlement and process payment."""
    claim_id = claim_record.get("claim_id", "N/A")
    amount = claim_record.get("amount", 0.0)
    log_step("7.7b", 5, f"[Payout] Processing settlement of ${amount:,.2f} for {claim_id}...")
    time.sleep(0.5)  # Simulate payment processing
    log_success(f"  Settlement of ${amount:,.2f} paid for claim {claim_id}.")
    return True


log_success("Insurance claims agents defined: intake, validator, classifier, payout.")

In [ ]:
# =============================================================================
# Section 7: Claims Workflow Manager — State Machine
# Ref: Section 7.7b, pp. 199–200 — Figure 7.5 and Table 7.2
#
# State transitions:
#   Intake -> Validating -> Assessing Risk -> Processing Payout -> Closed: Approved
#                |                 |                    |
#                v                 v                    v
#         Closed: Rejected   Pending Human Review  Closed: Rejected
#                             (approve -> Payout)    (payment failed)
#                             (reject  -> Rejected)
#
# Guard: confidence_score < 0.85 -> escalate to Pending Human Review
# =============================================================================

def claims_workflow_manager(
    claim: Dict[str, Any],
    policies: Dict,
) -> Dict[str, Any]:
    """Process a single insurance claim through the state machine.

    Returns an audit record with all state transitions and decisions.
    """
    claim_id = claim.get("claim_id", "N/A")
    audit_trail: List[Dict[str, str]] = []

    log_info("=" * 60)
    log_info(f"Claims Workflow: Processing {claim_id}")
    log_info("=" * 60)

    def log_transition(from_state: str, to_state: str, reason: str):
        entry = {"from": from_state, "to": to_state, "reason": reason}
        audit_trail.append(entry)
        log_info(f"  State: {from_state} -> {to_state} ({reason})")

    # --- State: Intake ---
    claim_record = intake_agent(claim)
    if claim_record is None:
        log_transition("Intake", "Closed: Rejected", "Intake agent failed")
        return {"claim_id": claim_id, "final_state": "Closed: Rejected", "audit": audit_trail}
    log_transition("Start", "Intake", "Form submitted")

    # --- State: Validating ---
    is_valid = validator_agent(claim_record, policies)
    if not is_valid:
        log_transition("Intake", "Closed: Rejected", "Validation failed")
        log_error(f"Claim {claim_id} REJECTED at validation.")
        return {"claim_id": claim_id, "final_state": "Closed: Rejected", "audit": audit_trail}
    log_transition("Intake", "Validating", "All fields populated")
    log_transition("Validating", "Assessing Risk", "Validation passed")

    # --- State: Assessing Risk ---
    classification = classifier_agent(claim_record)
    confidence = classification.get("confidence_score", 0.0)
    risk = classification.get("risk", "high")

    # Guard: confidence_score < 0.85 -> escalate (Table 7.2)
    if confidence < 0.85:
        log_transition("Assessing Risk", "Pending Human Review",
                       f"confidence_score {confidence} < 0.85 threshold")
        log_warning(f"Claim {claim_id} escalated for human review (confidence={confidence}).")

        # HITL: Escalation Agent
        log_step("7.7b", 4, f"[Escalation] Claim {claim_id} awaiting human decision...")
        if LIVE_MODE and INTERACTIVE_MODE:
            decision = ""
            while decision not in ["approve", "reject"]:
                decision = input("  Type 'approve' or 'reject': ").lower().strip()
        else:
            log_mock("  Auto-approving after 2s delay (Simulation Mode)...")
            time.sleep(2)
            decision = "approve"

        if decision == "reject":
            log_transition("Pending Human Review", "Closed: Rejected", "Human rejected")
            log_error(f"Claim {claim_id} REJECTED by human reviewer.")
            return {"claim_id": claim_id, "final_state": "Closed: Rejected", "audit": audit_trail}
        else:
            log_transition("Pending Human Review", "Processing Payout", "Human approved")
            log_success(f"Claim {claim_id} APPROVED by human reviewer.")
    else:
        log_transition("Assessing Risk", "Processing Payout",
                       f"Low risk, confidence {confidence} >= 0.85")

    # --- State: Processing Payout ---
    payment_ok = payout_agent(claim_record)
    if not payment_ok:
        log_transition("Processing Payout", "Closed: Rejected", "Payment failed")
        log_error(f"Claim {claim_id} REJECTED — payment failure.")
        return {"claim_id": claim_id, "final_state": "Closed: Rejected", "audit": audit_trail}

    log_transition("Processing Payout", "Closed: Approved", "Settlement paid")
    log_success(f"Claim {claim_id} APPROVED and settled.")

    return {"claim_id": claim_id, "final_state": "Closed: Approved", "audit": audit_trail}


log_success("claims_workflow_manager defined — full state machine with guard conditions.")

In [ ]:
# =============================================================================
# Section 7: Demo — Run All 3 Test Claims
# Ref: Strategy §6.4 — Expected outcomes:
#   CLM-4821: Auto-approved (confidence 0.91, water_damage, $8,400)
#   CLM-5099: Escalated then auto-approved (confidence 0.79, fire_damage, $47,000)
#   CLM-5100: Rejected at validation (expired policy)
# =============================================================================

log_info("=" * 60)
log_info("Section 7 Demo: Insurance Claims Workflow — 3 Test Claims")
log_info("=" * 60)

claim_results = {}
for claim in test_claims:
    result = claims_workflow_manager(claim, policy_db)
    claim_results[result["claim_id"]] = result
    print()

# --- Summary and Audit Trail ---
log_info("=" * 60)
log_info("Claims Processing Summary:")
for cid, result in claim_results.items():
    final = result["final_state"]
    badge_func = log_success if "Approved" in final else log_error
    badge_func(f"  {cid}: {final}")

print()
log_info("Audit Trails:")
for cid, result in claim_results.items():
    log_info(f"  {cid}:")
    for entry in result["audit"]:
        log_info(f"    {entry['from']} -> {entry['to']} ({entry['reason']})")

log_success("Section 7 complete. All 3 test claims processed with full audit trails.")

---
## Section 8: Summary and Key Takeaways

**Chapter 7** demonstrated the progression from foundational tool invocation to sophisticated
multi-agent orchestration and persistent workflow systems. We explored three architectural
patterns, each building on the last:

### Pattern 1: Tool-Using Agents (Sections 7.1–7.3)
A single agent extends its reasoning by invoking external functions through a **Think/Plan/Act**
cycle. The tool chest, tool registry, and execution engine work together to transform natural
language requests into concrete outputs. The **selection funnel** ensures reliable tool discovery,
and **safe invocation wrappers** provide resilience against the four categories of failure.

### Pattern 2: Chain-of-Agents Orchestrators (Sections 7.4–7.6)
Multiple specialized agents collaborate under a **cooperation protocol** with four pillars:
defined roles, a common communication infrastructure, shared memory, and execution
orchestration. **Episodic memory** preserves state across handoffs, and **conflict resolution**
mechanisms detect divergence between agent outputs using calibrated confidence scores.

### Pattern 3: Agentic Workflow Systems (Section 7.7)
Long-running business processes modeled as **state machines** with guard conditions,
**human-in-the-loop checkpoints**, and complete audit trails. The e-commerce and insurance
workflows demonstrated how to embed intelligent agents within deterministic process flows,
balancing automation speed with human governance.

### Cross-Cutting Principles
- **Fail-Gracefully Architecture:** `@graceful_fallback` on every function ensures no single
  failure crashes the system.
- **Visual Observability:** Color-coded logs provide a clear execution trail.
- **Simulation Mode:** `MockLLM` enables the entire notebook to run without any API key.

---

*These patterns provide the foundation for production-ready agent systems. In Chapter 8,
we apply these orchestration patterns to data analysis and reasoning agents that explore
datasets, recommend visualizations, and perform statistical reasoning.*

---
**Book:** *Agents* by Imran Ahmad (Packt, 2026 — B34135)